In [ ]:
# ============================================================
# CELL 1 - GPU CHECK
# ============================================================
#
# This project requires a CUDA-enabled GPU for Unsloth training.
# Google Colab provides access to NVIDIA GPUs such as T4/L4.
# We stop early if a GPU is not available.
# ============================================================

import sys
import torch

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not detected. In Colab go to "
        "Runtime > Change runtime type > Hardware accelerator > GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))

gpu_memory = (
    torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
)

print(f"GPU memory: {gpu_memory:.2f} GB")
print("\nGPU check PASSED.")

Python version: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB

GPU check PASSED.


In [ ]:
# ============================================================
# CELL 2 - INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install -q -U unsloth datasets trl peft accelerate bitsandbytes

In [ ]:
# ============================================================
# CELL 3 - IMPORT LIBRARIES
# ============================================================

import os
import json
import random
import torch

from datasets import Dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig

print("All required libraries imported successfully.")

All required libraries imported successfully.


In [ ]:
# ============================================================
# CELL 4 - PROJECT CONFIGURATION
# ============================================================

DOMAIN = "Indian income tax, GST, deductions and ITR filing"

MODEL_NAME = "unsloth/Qwen2.5-1.5B-bnb-4bit"

DATASET_SIZE = 60
MAX_SEQ_LENGTH = 2048

TRAIN_STEPS = 75
LEARNING_RATE = 2e-4

LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

SEED = 42

OUTPUT_DIR = "tax_training_output"
LORA_DIR = "tax_lora_model"
MERGED_DIR = "tax_model_merged"
GGUF_DIR = "tax_model_gguf"

random.seed(SEED)
torch.manual_seed(SEED)

print("Project configuration loaded.")
print()
print("Domain:", DOMAIN)
print("Base model:", MODEL_NAME)
print("Dataset size:", DATASET_SIZE)
print("Training steps:", TRAIN_STEPS)
print("Learning rate:", LEARNING_RATE)
print("LoRA rank:", LORA_RANK)

Project configuration loaded.

Domain: Indian income tax, GST, deductions and ITR filing
Base model: unsloth/Qwen2.5-1.5B-bnb-4bit
Dataset size: 60
Training steps: 75
Learning rate: 0.0002
LoRA rank: 16


In [ ]:
# ============================================================
# CELL 5 - SYNTHETIC DATASET GENERATION
# ============================================================
#
# The dataset contains instruction-response examples covering:
# income tax, ITR, TDS, GST, deductions, exemptions,
# salary income, Form 16, capital gains and related topics.
#
# The answers intentionally avoid hard-coded tax rates,
# thresholds and deadlines because these can change by year.
# ============================================================

seed_examples = [
    {
        "instruction": "What is income tax?",
        "output": (
            "Income tax is a tax imposed by the government on "
            "taxable income earned by individuals and other taxpayers."
        )
    },
    {
        "instruction": "What is an ITR?",
        "output": (
            "ITR stands for Income Tax Return. It is a form used "
            "by taxpayers to report income, deductions and tax "
            "information to the Income Tax Department."
        )
    },
    {
        "instruction": "What is Section 80C?",
        "output": (
            "Section 80C provides eligible individual taxpayers "
            "with deductions for certain qualifying investments "
            "and payments, subject to applicable rules and limits."
        )
    },
    {
        "instruction": "What is GST?",
        "output": (
            "GST stands for Goods and Services Tax. It is an "
            "indirect tax charged on the supply of goods and "
            "services in India."
        )
    },
    {
        "instruction": "What is HRA?",
        "output": (
            "HRA stands for House Rent Allowance. Eligible salaried "
            "individuals may be able to claim an exemption on "
            "part of their HRA subject to applicable rules."
        )
    },
    {
        "instruction": "What is TDS?",
        "output": (
            "TDS stands for Tax Deducted at Source. It is a system "
            "where tax is deducted when certain payments are made "
            "and deposited with the government."
        )
    },
    {
        "instruction": "What is Form 16?",
        "output": (
            "Form 16 is a certificate issued by an employer "
            "containing information about salary income and tax "
            "deducted at source."
        )
    },
    {
        "instruction": "What is a tax deduction?",
        "output": (
            "A tax deduction is an eligible amount that can be "
            "reduced from taxable income according to applicable "
            "tax rules."
        )
    },
    {
        "instruction": "What is a tax exemption?",
        "output": (
            "A tax exemption generally means that qualifying income "
            "or a qualifying portion of income is excluded from "
            "taxation subject to specified conditions."
        )
    },
    {
        "instruction": "What is advance tax?",
        "output": (
            "Advance tax is income tax paid during the financial "
            "year rather than entirely after the year has ended, "
            "subject to applicable conditions."
        )
    }
]


topics = [
    "salary income",
    "ITR filing",
    "income tax deductions",
    "income tax exemptions",
    "TDS",
    "Form 16",
    "GST",
    "capital gains",
    "advance tax",
    "tax documents",
    "old and new tax regimes",
    "rental income"
]

question_templates = [
    "What is {topic}?",
    "Explain {topic} in simple terms.",
    "Why is {topic} important for taxpayers?",
    "What should a taxpayer know about {topic}?",
    "What is the purpose of {topic}?"
]


def create_answer(topic):
    return (
        f"{topic.title()} is an area of Indian taxation that "
        f"taxpayers may need to understand when calculating or "
        f"reporting their tax obligations. The exact treatment "
        f"depends on the applicable tax rules and financial year."
    )


tax_data = []

# Add seed examples
tax_data.extend(seed_examples)

# Generate additional examples
for topic in topics:
    for template in question_templates:

        if len(tax_data) >= DATASET_SIZE:
            break

        question = template.format(topic=topic)

        tax_data.append({
            "instruction": question,
            "output": create_answer(topic)
        })

    if len(tax_data) >= DATASET_SIZE:
        break


# Remove duplicate questions
unique_examples = {}

for example in tax_data:

    question_key = example["instruction"].strip().lower()

    if question_key not in unique_examples:
        unique_examples[question_key] = example


tax_data = list(unique_examples.values())[:DATASET_SIZE]


print("=" * 70)
print("SYNTHETIC DATASET CREATED")
print("=" * 70)

print("Number of examples:", len(tax_data))

if len(tax_data) < 50:
    raise ValueError(
        "Dataset contains fewer than 50 examples."
    )

SYNTHETIC DATASET CREATED
Number of examples: 56


In [ ]:
# ============================================================
# CELL 6 - DISPLAY SAMPLE DATA
# ============================================================

print("\nSample training examples:\n")

for number, example in enumerate(tax_data[:5], start=1):

    print("-" * 70)
    print(f"Example {number}")
    print()
    print("Instruction:")
    print(example["instruction"])
    print()
    print("Response:")
    print(example["output"])


Sample training examples:

----------------------------------------------------------------------
Example 1

Instruction:
What is income tax?

Response:
Income tax is a tax imposed by the government on taxable income earned by individuals and other taxpayers.
----------------------------------------------------------------------
Example 2

Instruction:
What is an ITR?

Response:
ITR stands for Income Tax Return. It is a form used by taxpayers to report income, deductions and tax information to the Income Tax Department.
----------------------------------------------------------------------
Example 3

Instruction:
What is Section 80C?

Response:
Section 80C provides eligible individual taxpayers with deductions for certain qualifying investments and payments, subject to applicable rules and limits.
----------------------------------------------------------------------
Example 4

Instruction:
What is GST?

Response:
GST stands for Goods and Services Tax. It is an indirect tax charged on

In [ ]:
# ============================================================
# CELL 7 - SAVE DATASET
# ============================================================

DATASET_FILE = "tax_dataset.json"

with open(DATASET_FILE, "w", encoding="utf-8") as file:
    json.dump(
        tax_data,
        file,
        indent=2,
        ensure_ascii=False
    )

print(f"Dataset saved successfully: {DATASET_FILE}")
print("File size:", os.path.getsize(DATASET_FILE), "bytes")

Dataset saved successfully: tax_dataset.json
File size: 15620 bytes


In [ ]:
# ============================================================
# CELL 8 - CONVERT DATASET TO CHAT FORMAT
# ============================================================

def convert_to_chat_format(example):

    return {
        "messages": [
            {
                "role": "user",
                "content": example["instruction"]
            },
            {
                "role": "assistant",
                "content": example["output"]
            }
        ]
    }


chat_data = [
    convert_to_chat_format(example)
    for example in tax_data
]

dataset = Dataset.from_list(chat_data)

print(dataset)

print("\nFirst training example:")
print(dataset[0])

Dataset({
    features: ['messages'],
    num_rows: 56
})

First training example:
{'messages': [{'role': 'user', 'content': 'What is income tax?'}, {'role': 'assistant', 'content': 'Income tax is a tax imposed by the government on taxable income earned by individuals and other taxpayers.'}]}


In [ ]:
# ============================================================
# CELL 9 - TRAIN / TEST SPLIT
# ============================================================

split_dataset = dataset.train_test_split(
    test_size=0.1,
    seed=SEED
)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

print("Training examples:", len(train_dataset))
print("Test examples:", len(test_dataset))

Training examples: 50
Test examples: 6


In [ ]:
# ============================================================
# CELL 10 - LOAD QWEN MODEL
# ============================================================

print("Loading Qwen2.5 1.5B 4-bit model...")
print("This may take a few minutes.\n")


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None
)


print("\nModel loaded successfully.")
print("Model:", MODEL_NAME)

Loading Qwen2.5 1.5B 4-bit model...
This may take a few minutes.

==((====))==  Unsloth 2026.9.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


Model loaded successfully.
Model: unsloth/Qwen2.5-1.5B-bnb-4bit


In [ ]:
# ============================================================
# CELL 11 - APPLY LoRA
# ============================================================

model = FastLanguageModel.get_peft_model(
    model,

    r=LORA_RANK,

    lora_alpha=LORA_ALPHA,

    lora_dropout=LORA_DROPOUT,

    bias="none",

    use_gradient_checkpointing="unsloth",

    random_state=SEED,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

print("LoRA configuration applied successfully.")

LoRA configuration applied successfully.


In [ ]:
# ============================================================
# CELL 12 - VERIFY TRAINABLE PARAMETERS
# ============================================================

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_percentage = (
    100 * trainable_parameters / total_parameters
)

print("=" * 70)
print("LORA PARAMETER CHECK")
print("=" * 70)

print(
    f"Trainable parameters : {trainable_parameters:,}"
)

print(
    f"Total parameters     : {total_parameters:,}"
)

print(
    f"Trainable percentage  : {trainable_percentage:.2f}%"
)

if trainable_percentage >= 5:

    raise ValueError(
        "More than 5% of parameters are trainable."
    )

print("\nPASS: Trainable parameters are below 5%.")

LORA PARAMETER CHECK
Trainable parameters : 18,464,768
Total parameters     : 907,081,216
Trainable percentage  : 2.04%

PASS: Trainable parameters are below 5%.


In [ ]:
# ============================================================
# CELL 13 - TRAINING CONFIGURATION
# ============================================================

use_bf16 = is_bfloat16_supported()

training_config = SFTConfig(

    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=LEARNING_RATE,

    max_steps=TRAIN_STEPS,

    logging_steps=10,

    save_steps=50,

    warmup_steps=5,

    weight_decay=0.01,

    optim="adamw_8bit",

    fp16=not use_bf16,

    bf16=use_bf16,

    report_to="none",

    seed=SEED,

    max_length=MAX_SEQ_LENGTH,

    packing=False,

    eos_token="<|im_end|>"
)


print("Training configuration created.")
print()
print("Training steps:", TRAIN_STEPS)
print("Learning rate:", LEARNING_RATE)
print("Batch size:", 2)
print("Gradient accumulation:", 4)

Training configuration created.

Training steps: 75
Learning rate: 0.0002
Batch size: 2
Gradient accumulation: 4


In [ ]:
# ============================================================
# CELL 14 - CREATE SFT TRAINER
# ============================================================

def formatting_func(example):
    """
    Convert the training example into plain text.

    Unsloth may pass either a single example or a batch,
    so this function handles both cases.
    """

    messages = example["messages"]

    # --------------------------------------------------------
    # CASE 1: Single example
    # --------------------------------------------------------
    if isinstance(messages, list) and len(messages) > 0:

        # Single example:
        # [
        #   {"role": "user", "content": "..."},
        #   {"role": "assistant", "content": "..."}
        # ]

        if isinstance(messages[0], dict):

            user_message = messages[0]["content"]
            assistant_message = messages[1]["content"]

            text = (
                "### User:\n"
                + user_message
                + "\n\n"
                + "### Assistant:\n"
                + assistant_message
            )

            return [text]

    # --------------------------------------------------------
    # CASE 2: Batch of examples
    # --------------------------------------------------------
    if isinstance(messages, list):

        formatted_texts = []

        for message_group in messages:

            if not isinstance(message_group, list):
                continue

            if len(message_group) < 2:
                continue

            user_message = message_group[0]["content"]
            assistant_message = message_group[1]["content"]

            text = (
                "### User:\n"
                + user_message
                + "\n\n"
                + "### Assistant:\n"
                + assistant_message
            )

            formatted_texts.append(text)

        return formatted_texts

    raise ValueError(
        "Unexpected dataset format received by formatting_func."
    )


trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    formatting_func=formatting_func,
    args=training_config
)

print("SFTTrainer created successfully.")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/50 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/6 [00:00<?, ? examples/s]

SFTTrainer created successfully.


In [ ]:
# ============================================================
# CELL 15 - TRAIN MODEL
# ============================================================

print("=" * 70)
print("STARTING LORA FINE-TUNING")
print("=" * 70)

print("\nTraining the model...")
print("Please wait. Training may take several minutes.\n")

training_result = trainer.train()

print("\n" + "=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)

print(f"Final training loss: {training_result.training_loss:.4f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None}.


STARTING LORA FINE-TUNING

Training the model...
Please wait. Training may take several minutes.



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 11 | Total steps = 75
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss
10,2.448527
20,0.508172
30,0.198499
40,0.132458
50,0.105021
60,0.098749
70,0.094065



TRAINING COMPLETED
Final training loss: 0.4842


In [ ]:
# ============================================================
# CELL 16 - TRAINING METRICS
# ============================================================

print("=" * 70)
print("TRAINING METRICS")
print("=" * 70)

for key, value in training_result.metrics.items():
    print(f"{key}: {value}")

TRAINING METRICS
train_runtime: 160.7383
train_samples_per_second: 3.733
train_steps_per_second: 0.467
total_flos: 217833722898432.0
train_loss: 0.48421259204546613
epoch: 10.8


In [ ]:
# ============================================================
# CELL 17 - SAVE LORA ADAPTER
# ============================================================

import os

os.makedirs(LORA_DIR, exist_ok=True)

model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)

print("LoRA adapter saved successfully.")
print("Location:", LORA_DIR)

print("\nSaved files:")

for filename in os.listdir(LORA_DIR):
    print(" -", filename)

LoRA adapter saved successfully.
Location: tax_lora_model

Saved files:
 - README.md
 - adapter_config.json
 - adapter_model.safetensors
 - tokenizer.json
 - tokenizer_config.json


In [ ]:
# ============================================================
# CELL 18 - MERGE LORA WITH BASE MODEL
# ============================================================

print("=" * 70)
print("CREATING MERGED MODEL")
print("=" * 70)

model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit"
)

print("\nMerged model saved successfully.")
print("Location:", MERGED_DIR)

CREATING MERGED MODEL
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 5108.77it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:01<00:00, 61.80s/it]


Unsloth: Merge process complete. Saved to `/content/tax_model_merged`

Merged model saved successfully.
Location: tax_model_merged


In [ ]:
# ============================================================
# CELL 19 - EXPORT GGUF
# ============================================================

print("=" * 70)
print("EXPORTING GGUF MODEL")
print("=" * 70)

print("Quantization: Q4_K_M")
print()

model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m"
)

print("\nGGUF export completed.")
print("Location:", GGUF_DIR)

EXPORTING GGUF MODEL
Quantization: Q4_K_M

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 7037.42it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:57<00:00, 57.13s/it]


Unsloth: Merge process complete. Saved to `/content/tax_model_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['tax_model_gguf_gguf/Qwen2.5-1.5B.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['tax_model_gguf_gguf/Qwen2.5-1.5B.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'u

In [ ]:
# ============================================================
# CELL 20 - FIND AND VERIFY GGUF
# ============================================================

gguf_files = []

for root, directories, files in os.walk(GGUF_DIR):
    for filename in files:

        if filename.lower().endswith(".gguf"):

            full_path = os.path.join(
                root,
                filename
            )

            gguf_files.append(full_path)


print("=" * 70)
print("GGUF FILES")
print("=" * 70)

if not gguf_files:

    raise FileNotFoundError(
        "No GGUF file was found."
    )


for file_path in gguf_files:

    size_mb = (
        os.path.getsize(file_path)
        / (1024 ** 2)
    )

    print(f"\nFile: {file_path}")
    print(f"Size: {size_mb:.1f} MB")

GGUF FILES


FileNotFoundError: No GGUF file was found.

In [ ]:
# ============================================================
# DIAGNOSTIC - CHECK GGUF EXPORT DIRECTORY
# ============================================================

import os

print("GGUF_DIR:", GGUF_DIR)
print("Directory exists:", os.path.exists(GGUF_DIR))

if os.path.exists(GGUF_DIR):
    print("\nFiles and folders inside GGUF_DIR:")

    for root, directories, files in os.walk(GGUF_DIR):
        print("\nFolder:", root)

        for filename in files:
            print("  -", filename)
else:
    print("\nGGUF directory does not exist.")

GGUF_DIR: tax_model_gguf
Directory exists: True

Files and folders inside GGUF_DIR:

Folder: tax_model_gguf
  - generation_config.json
  - model.safetensors
  - config.json
  - tokenizer.json
  - tokenizer_config.json

Folder: tax_model_gguf/.cache

Folder: tax_model_gguf/.cache/huggingface
  - .gitignore
  - CACHEDIR.TAG

Folder: tax_model_gguf/.cache/huggingface/download
  - tokenizer.model.lock
  - model.safetensors.metadata
  - model.safetensors.lock


In [ ]:
# ============================================================
# CELL 20 - FIND AND VERIFY GGUF
# ============================================================

import os

# Unsloth created the GGUF in this directory
GGUF_OUTPUT_DIR = "tax_model_gguf_gguf"

gguf_files = []

for root, directories, files in os.walk(GGUF_OUTPUT_DIR):

    for filename in files:

        if filename.lower().endswith(".gguf"):

            full_path = os.path.join(
                root,
                filename
            )

            gguf_files.append(full_path)


print("=" * 70)
print("GGUF FILES")
print("=" * 70)

if not gguf_files:
    raise FileNotFoundError(
        "No GGUF file was found."
    )

for file_path in gguf_files:

    size_mb = os.path.getsize(file_path) / (1024 ** 2)

    print("\nFile:", file_path)
    print(f"Size: {size_mb:.1f} MB")

print("\nGGUF verification completed successfully.")

GGUF FILES

File: tax_model_gguf_gguf/Qwen2.5-1.5B.Q4_K_M.gguf
Size: 940.4 MB

GGUF verification completed successfully.


In [ ]:
# ============================================================
# CELL 21 - DOWNLOAD GGUF TO YOUR COMPUTER
# ============================================================

from google.colab import files

gguf_file = gguf_files[0]

print("Downloading:")
print(gguf_file)

files.download(gguf_file)

Downloading:
tax_model_gguf_gguf/Qwen2.5-1.5B.Q4_K_M.gguf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>